# Descriptive Figures

In [5]:
libraries <- c("dplyr", "magrittr", "tidyr", "ggplot2", "ggpubr", "scales", "readr", "sf", "geodata", "forcats")
for (pkg in libraries) {
  library(pkg, character.only = TRUE, warn.conflicts = FALSE, quietly = TRUE)
}

DATA_FILE <- "../data/typhoon_mortality_dummy.csv"
FIG_DIR <- "../output/fig"
Sys.setenv(PROJ_DATA = "/opt/conda/share/proj", GDAL_DATA = "/opt/conda/share/gdal")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)

source("../src/utils.R")


There appears to be a problem with the PROJ installation



terra 1.9.1




Attaching package: ‘terra’




The following object is masked from ‘package:scales’:

    rescale




The following object is masked from ‘package:ggpubr’:

    rotate




The following object is masked from ‘package:tidyr’:

    extract




The following objects are masked from ‘package:magrittr’:

    extract, inset




## Prepare data

In [7]:
df_raw <- read.csv(DATA_FILE)

df <- df_raw %>%
  mutate(
    pref = factor(pref),
    region_id = as.integer(pref),
    week_id = as.integer(factor(week, levels = unique(week))),
    year = factor(year),
    y_young = as.numeric(totdeath_all) - (as.numeric(totdeath_age7079) + as.numeric(totdeath_age8089) + as.numeric(totdeath_age9099)),
    pop_young = (as.numeric(totpop_all) - (as.numeric(totpop_7079) + as.numeric(totpop_8099))) * 1000,
    y_eld = as.numeric(totdeath_age7079) + as.numeric(totdeath_age8089) + as.numeric(totdeath_age9099),
    pop_eld = (as.numeric(totpop_7079) + as.numeric(totpop_8099)) * 1000,
    holiday = as.integer(hol),
    tmean = as.numeric(tmean),
    pwind = as.numeric(pwind),
    pwind_exp = if_else(pwind >= 17.2, pwind, 0)
  )

df <- df %>%
  arrange(region_id, week) %>%
  group_by(region_id) %>%
  mutate(
    exp0 = pwind_exp,
    exp1 = lag(pwind_exp, 1, default = 0),
    exp2 = lag(pwind_exp, 2, default = 0)
  ) %>%
  ungroup()

soc <- df %>%
  group_by(region_id) %>%
  slice(1) %>%
  ungroup() %>%
  select(region_id, income, flood_per, landslide_per, med4_ratio) %>%
  mutate(across(-region_id, ~ as.numeric(scale(.))))

df <- df %>%
  select(-income, -flood_per, -landslide_per, -med4_ratio) %>%
  left_join(soc, by = "region_id")

j <- get_jpn_pref_sf(path = "../data/gadm_cache")
jpn <- j$jpn
jpn_id <- j$jpn_id
pref_lookup <- get_english_pref_lookup(jpn)

sf_to_polygon_df <- function(sf_obj) {
  sf_obj <- sf::st_cast(sf_obj, "MULTIPOLYGON")
  coords <- sf::st_coordinates(sf_obj)
  level_cols <- grep("^L[0-9]+$", colnames(coords), value = TRUE)
  feature_col <- tail(level_cols, 1)
  attrs <- sf::st_drop_geometry(sf_obj)
  out <- cbind(as.data.frame(coords), attrs[coords[, feature_col], , drop = FALSE])
  out$group_id <- interaction(out[, level_cols, drop = FALSE], drop = TRUE)
  out
}

df_plot <- df %>%
  mutate(
    date = as.Date(date),
    pref_en = factor(pref_lookup$prefecture_en[region_id], levels = pref_lookup$prefecture_en),
    exposure = if_else(pwind >= 17.2, 1L, 0L)
  )


## Descriptive overview figure

In [8]:
cum_exposed <- df_plot %>%
  group_by(region_id) %>%
  summarise(total_exposed_weeks = sum(exposure, na.rm = TRUE), .groups = "drop")

map_exposed <- jpn_id %>% left_join(cum_exposed, by = "region_id")
map_exposed_df <- sf_to_polygon_df(map_exposed)

p_exp_weeks <- ggplot(map_exposed_df, aes(x = X, y = Y, group = group_id, fill = total_exposed_weeks)) +
  geom_polygon(colour = "white", linewidth = 0.2) +
  coord_equal() +
  scale_fill_viridis_c(name = "", option = "C", breaks = pretty_breaks(5), labels = comma) +
  guides(fill = guide_colorbar(direction = "vertical", barwidth = grid::unit(1, "cm"), barheight = grid::unit(6, "cm"), title.position = "top")) +
  labs(title = "Cumulative number of tropical cyclone-exposed weeks by prefecture, 2010-2019") +
  theme_minimal(base_size = 14) +
  theme(
    text = element_text(size = 20, family = "sans", colour = "black"),
    axis.text = element_text(size = 14, family = "sans", colour = "black"),
    axis.title = element_blank(),
    legend.position.inside = c(0.9, 0.4),
    legend.text = element_text(size = 14)
  )

p_temp <- ggplot(df_plot, aes(x = date, y = forcats::fct_rev(pref_en), fill = tmean)) +
  geom_tile() +
  scale_fill_viridis_c(name = "Temperature (C)", option = "C", breaks = pretty_breaks(5)) +
  scale_x_date(date_breaks = "3 months", date_labels = "%b\n%Y", expand = c(0, 0)) +
  labs(x = NULL, y = NULL, title = "Mean temperature") +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "right",
    legend.text = element_text(size = 14),
    text = element_text(size = 20, family = "sans", colour = "black"),
    axis.text.x = element_text(angle = 45, hjust = 1),
    axis.text = element_text(size = 14, family = "sans", colour = "black")
  )

p_wind <- ggplot(df_plot, aes(x = date, y = forcats::fct_rev(pref_en), fill = pwind)) +
  geom_tile() +
  scale_fill_viridis_c(name = "Windspeed (m/s)", option = "C", breaks = pretty_breaks(5)) +
  scale_x_date(date_breaks = "3 months", date_labels = "%b\n%Y", expand = c(0, 0)) +
  labs(x = "Date", y = NULL, title = "Windspeed") +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "right",
    legend.text = element_text(size = 14),
    text = element_text(size = 20, family = "sans", colour = "black"),
    axis.text.x = element_text(angle = 45, hjust = 1),
    axis.text = element_text(size = 14, family = "sans", colour = "black")
  )

temp_wind_panel <- ggpubr::ggarrange(
  p_temp,
  p_wind,
  ncol = 1,
  nrow = 2,
  heights = c(1, 1),
  common.legend = FALSE,
  labels = c("(B)", "(C)"),
  font.label = list(size = 22)
)

descriptive1_plot <- ggpubr::ggarrange(
  p_exp_weeks,
  temp_wind_panel,
  ncol = 2,
  nrow = 1,
  common.legend = FALSE,
  labels = c("(A)", ""),
  font.label = list(size = 22)
)

descriptive1_path <- file.path(FIG_DIR, "figure1.pdf")
ggsave(filename = descriptive1_path, plot = descriptive1_plot,  width = 48, height = 18, units = "in", device = grDevices::cairo_pdf)
descriptive1_path


[1] "../output/fig/figure1.pdf"

## Social indicator maps

In [4]:
soc_map <- df_raw %>%
  mutate(region_id = as.integer(pref)) %>%
  group_by(region_id) %>%
  summarise(
    income = first(as.numeric(income)),
    flood_per = first(as.numeric(flood_per)),
    landslide_per = first(as.numeric(landslide_per)),
    med4ratio = first(as.numeric(med4_ratio)),
    .groups = "drop"
  )

plot_soc <- function(var, title_txt) {
  map_df <- jpn_id %>% left_join(soc_map %>% select(region_id, all_of(var)), by = "region_id")
  map_poly <- sf_to_polygon_df(map_df)
  ggplot(map_poly, aes(x = X, y = Y, group = group_id, fill = .data[[var]])) +
    geom_polygon(colour = "white", linewidth = 0.2) +
    coord_equal() +
    scale_fill_viridis_c(name = "", option = "C", breaks = pretty_breaks(5), labels = comma) +
    guides(fill = guide_colorbar(direction = "vertical", barwidth = grid::unit(1, "cm"), barheight = grid::unit(6, "cm"), title.position = "top")) +
    labs(title = title_txt) +
    theme_minimal(base_size = 14) +
    theme(
      text = element_text(size = 20, family = "sans", colour = "black"),
      axis.text = element_text(size = 14, family = "sans", colour = "black"),
      axis.title = element_blank(),
      legend.position.inside = c(0.9, 0.4),
      legend.text = element_text(size = 14)
    )
}

p_income <- plot_soc("income", "Prefecture-level income")
p_flood <- plot_soc("flood_per", "Potential flood risk (%)")
p_landslide <- plot_soc("landslide_per", "Potential landslide risk (%)")
p_med4 <- plot_soc("med4ratio", "Access to medical facilities")

descriptive_plot <- ggpubr::ggarrange(
  p_income,
  p_flood,
  p_landslide,
  p_med4,
  ncol = 2,
  nrow = 2,
  common.legend = FALSE,
  labels = c("(A)", "(B)", "(C)", "(D)"),
  font.label = list(size = 22)
)

descriptive_path <- file.path(FIG_DIR, "supple1.pdf")
ggsave(filename = descriptive_path, plot = descriptive_plot, width = 16, height = 16, units = "in", device = grDevices::cairo_pdf)
descriptive_path


[1] "../output/fig/descriptive.pdf"

In [5]:
list(
  descriptive1 = descriptive1_path,
  descriptive = descriptive_path
)


$descriptive1
[1] "../output/fig/descriptive1.pdf"

$descriptive
[1] "../output/fig/descriptive.pdf"